# Trace and evaluate MCP tool calls with TruLens

This cookbook builds a small LangGraph agent backed by a local Model Context Protocol (MCP) server. TruLens records the agent workflow and exposes each MCP call with its server name, tool name, arguments, result, latency, and error status.

The MCP server is deterministic and runs locally over stdio. An OpenAI API key is used only by the agent and the two LLM-as-a-judge metrics.

## Install dependencies

Restart the kernel after installing packages into an existing notebook environment.

In [ ]:
%pip install -q trulens-apps-langgraph trulens-providers-openai \
    langchain-openai mcp

## Configure the model

Set `OPENAI_API_KEY` in your environment before running this notebook. Do not put secrets directly in notebook cells.

In [ ]:
import os

assert os.environ.get("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY in your environment before continuing."
)

## Connect to the local MCP server

The official MCP Python SDK starts `mcp_weather_server.py` over stdio, creates a `ClientSession`, and discovers its tools. Keeping the server local makes this example reproducible and avoids additional service credentials.

In [ ]:
import sys
from contextlib import AsyncExitStack
from pathlib import Path

from mcp import ClientSession
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

server_path = Path.cwd() / "mcp_weather_server.py"
if not server_path.exists():
    server_path = Path.cwd() / "examples/cookbooks/mcp_weather_server.py"
assert server_path.exists(), f"Could not find {server_path.name}"

server_parameters = StdioServerParameters(
    command=sys.executable,
    args=[str(server_path.resolve())],
)
mcp_stack = AsyncExitStack()
read_stream, write_stream = await mcp_stack.enter_async_context(
    stdio_client(server_parameters)
)
mcp_session = await mcp_stack.enter_async_context(
    ClientSession(read_stream, write_stream)
)
await mcp_session.initialize()
available_tools = (await mcp_session.list_tools()).tools
[(tool.name, tool.description) for tool in available_tools]

## Build a tool-using LangGraph agent

The model decides which MCP tool to call. `ToolNode` executes the selected tool and routes its result back to the model for the final response.

In [ ]:
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langgraph.graph import START
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes


def mcp_attributes(ret, exception, *args, **kwargs):
    return {
        SpanAttributes.MCP.TOOL_NAME: kwargs["tool_name"],
        SpanAttributes.MCP.SERVER_NAME: "local-weather",
        SpanAttributes.MCP.INPUT_ARGUMENTS: str(kwargs["arguments"]),
        SpanAttributes.MCP.OUTPUT_CONTENT: str(ret or exception),
        SpanAttributes.MCP.OUTPUT_IS_ERROR: exception is not None,
    }


@instrument(
    span_type=SpanAttributes.SpanType.MCP,
    attributes=mcp_attributes,
)
async def call_mcp_tool(*, tool_name: str, arguments: dict) -> str:
    result = await mcp_session.call_tool(tool_name, arguments)
    text_parts = [
        content.text
        for content in result.content
        if getattr(content, "type", None) == "text"
    ]
    return "\n".join(text_parts)


def as_langchain_tool(mcp_tool):
    async def invoke(**arguments):
        return await call_mcp_tool(
            tool_name=mcp_tool.name, arguments=arguments
        )

    return StructuredTool.from_function(
        coroutine=invoke,
        name=mcp_tool.name,
        description=mcp_tool.description or "",
        args_schema=mcp_tool.inputSchema,
    )


tools = [as_langchain_tool(tool) for tool in available_tools]

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
model_with_tools = model.bind_tools(tools)


def call_model(state: MessagesState):
    return {"messages": [model_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "call_model")
builder.add_conditional_edges("call_model", tools_condition)
builder.add_edge("tools", "call_model")
graph = builder.compile()

## Define tool-quality metrics

Both metrics receive the complete trace. **Tool Selection** checks whether the agent chose an appropriate tool, while **Tool Calling** checks the arguments and result of the call.

In [ ]:
from trulens.core import Metric
from trulens.core import Selector
from trulens.providers.openai import OpenAI

provider = OpenAI(model_engine="gpt-4.1-mini")

tool_selection = Metric(
    implementation=provider.tool_selection_with_cot_reasons,
    name="Tool Selection",
).on({"trace": Selector(trace_level=True)})

tool_calling = Metric(
    implementation=provider.tool_calling_with_cot_reasons,
    name="Tool Calling",
).on({"trace": Selector(trace_level=True)})

## Record an agent run

The instrumented `call_mcp_tool` wrapper records each official SDK call as an MCP span. The span includes `ai.observability.mcp.tool_name`, `server_name`, `input_arguments`, `output_content`, `output_is_error`, and `execution_time_ms`.

In [ ]:
from trulens.apps.langgraph import TruGraph
from trulens.core import TruSession

session = TruSession()
tru_graph = TruGraph(
    app=graph,
    app_name="mcp-weather-agent",
    app_version="v1",
    feedbacks=[tool_selection, tool_calling],
)

question = "What is the weather in Chicago in Celsius?"
try:
    with tru_graph:
        response = await graph.ainvoke(
            {"messages": [("user", question)]}
        )
finally:
    await mcp_stack.aclose()

print(response["messages"][-1].content)

The expected trace contains two MCP calls: `get_weather` to retrieve Chicago's Fahrenheit observation and `convert_temperature` to convert it to Celsius. Try other supported cities or ask only for a conversion to compare tool-selection behavior.

## Inspect the trace and evaluations

Launch the dashboard, open the **mcp-weather-agent** record, and expand the trace. MCP spans are labeled with the tool name. Select a span to inspect its server, arguments, output, duration, and error status. The record also shows the Tool Selection and Tool Calling scores when evaluation completes.

In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session)